**Name: Rishi Raj Phadale**  
**Roll No.: 23102C0070**  
**Subject: R-Programming**  

## Experiment 3: Multi-Source Retail Sales Data Integration and Analysis


## 0. Environment Setup
Install and load all R packages required for import, wrangling, and SQL connectivity.

In [3]:
install.packages(c('readr','jsonlite','readxl','writexl','dplyr','tidyr',
                     'lubridate','DBI','RSQLite','stringr'))

suppressPackageStartupMessages({
  library(readr)      # CSV import
  library(jsonlite)   # JSON import
  library(readxl)     # Excel import
  library(writexl)    # Excel export
  library(dplyr)      # data manipulation
  library(tidyr)      # data cleaning
  library(lubridate)  # date handling
  library(DBI)        # database interface
  library(RSQLite)    # SQLite driver
  library(stringr)    # string cleaning
})

options(scipen = 999)
set.seed(123)

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



## 1. Data Source: UCI "Online Retail" Dataset
Obtain the **real** UCI Online Retail transactional dataset (Chen, D. (2015). *Online Retail* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33) and split it into the three required source files: `transactions.csv`, `products.json`, `customers.xlsx`.

In [2]:
uci_url <- 'https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx'

raw <- tryCatch({
  destfile <- 'Online_Retail_raw.xlsx'
  download.file(uci_url, destfile, mode = 'wb', quiet = TRUE)
  read_excel(destfile)
}, error = function(e) {
  message('Falling back to onlineretail mirror package (same UCI dataset)...')
  if (!require('onlineretail', quietly = TRUE)) {
    install.packages('remotes', repos = 'https://cloud.r-project.org')
    remotes::install_github('allanvc/onlineretail')
  }
  library(onlineretail)
  data('onlineretail')
  onlineretail
})

cat('Raw UCI dataset dimensions:', dim(raw), '\n')

Raw UCI dataset dimensions: 541909 8 


In [4]:
transactions_raw <- raw %>%
  select(InvoiceNo, StockCode, CustomerID, Quantity, InvoiceDate) %>%
  mutate(InvoiceDate = format(InvoiceDate, '%Y-%m-%d %H:%M:%S'))
write_csv(transactions_raw, 'transactions.csv')

products_raw <- raw %>%
  distinct(StockCode, .keep_all = TRUE) %>%
  select(StockCode, Description, UnitPrice)
write(toJSON(products_raw, pretty = TRUE, auto_unbox = TRUE), 'products.json')

customers_raw <- raw %>%
  filter(!is.na(CustomerID)) %>%
  distinct(CustomerID, .keep_all = TRUE) %>%
  select(CustomerID, Country)
write_xlsx(customers_raw, 'customers.xlsx')

cat('transactions.csv :', nrow(transactions_raw), 'rows\n')
cat('products.json    :', nrow(products_raw), 'unique products\n')
cat('customers.xlsx   :', nrow(customers_raw), 'unique customers\n')

transactions.csv : 541909 rows
products.json    : 4070 unique products
customers.xlsx   : 4372 unique customers


## Task 1: Import and Clean the Data
Import the CSV, JSON, and Excel sources into R using `readr`, `jsonlite`, and `readxl`.

In [5]:
transactions <- read_csv('transactions.csv', show_col_types = FALSE)
products     <- fromJSON('products.json')
customers    <- read_excel('customers.xlsx')

head(transactions, 5)
head(products, 5)
head(customers, 5)

InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate
<chr>,<chr>,<dbl>,<dbl>,<dttm>
536365,85123A,17850,6,2010-12-01 08:26:00
536365,71053,17850,6,2010-12-01 08:26:00
536365,84406B,17850,8,2010-12-01 08:26:00
536365,84029G,17850,6,2010-12-01 08:26:00
536365,84029E,17850,6,2010-12-01 08:26:00


,StockCode,Description,UnitPrice
,<chr>,<chr>,<dbl>
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2.55
2,71053,WHITE METAL LANTERN,3.39
3,84406B,CREAM CUPID HEARTS COAT HANGER,2.75
4,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,3.39
5,84029E,RED WOOLLY HOTTIE WHITE HEART.,3.39


CustomerID,Country
<dbl>,<chr>
17850,United Kingdom
13047,United Kingdom
12583,France
13748,United Kingdom
15100,United Kingdom


### 1.1 Inspect the Data
Check dimensions, structure, and missing-value counts for each dataset before cleaning.

In [6]:
cat('transactions:', dim(transactions), '\n')
cat('products    :', dim(products), '\n')
cat('customers   :', dim(customers), '\n\n')

cat('Missing values per column - transactions:\n'); print(colSums(is.na(transactions)))
cat('\nMissing values per column - products:\n');     print(colSums(is.na(products)))
cat('\nMissing values per column - customers:\n');    print(colSums(is.na(customers)))

cat('\nDuplicate transaction rows:', sum(duplicated(transactions)), '\n')
cat('Zero-quantity rows:', sum(transactions$Quantity == 0, na.rm = TRUE), '\n')
cat('Negative-quantity rows (cancellations/returns):', sum(transactions$Quantity < 0, na.rm = TRUE), '\n')
cat('Zero/negative UnitPrice in products:', sum(products$UnitPrice <= 0, na.rm = TRUE), '\n')

transactions: 541909 5 
products    : 4070 3 
customers   : 4372 2 

Missing values per column - transactions:
  InvoiceNo   StockCode  CustomerID    Quantity InvoiceDate 
          0           0      135080           0           0 

Missing values per column - products:
  StockCode Description   UnitPrice 
          0         176           0 

Missing values per column - customers:
CustomerID    Country 
         0          0 

Duplicate transaction rows: 5429 
Zero-quantity rows: 0 
Negative-quantity rows (cancellations/returns): 10624 
Zero/negative UnitPrice in products: 215 


### 1.2 Clean the Data
Remove duplicates, drop rows with missing `CustomerID`, and filter out invalid `Quantity` and `UnitPrice` values.

In [7]:
transactions_clean <- transactions %>%
  distinct() %>%
  filter(!is.na(CustomerID)) %>%
  filter(Quantity > 0) %>%
  mutate(InvoiceDate = as.POSIXct(InvoiceDate, format = '%Y-%m-%d %H:%M:%S'))

products_clean <- products %>%
  distinct(StockCode, .keep_all = TRUE) %>%
  filter(!is.na(UnitPrice), UnitPrice > 0) %>%
  filter(!is.na(Description))

customers_clean <- customers %>%
  distinct(CustomerID, .keep_all = TRUE) %>%
  filter(!is.na(Country))

cat('Rows before cleaning (transactions):', nrow(transactions), '\n')
cat('Rows after cleaning  (transactions):', nrow(transactions_clean), '\n')
cat('Products retained:', nrow(products_clean), 'of', nrow(products), '\n')
cat('Customers retained:', nrow(customers_clean), 'of', nrow(customers), '\n')

Rows before cleaning (transactions): 541909 
Rows after cleaning  (transactions): 392708 
Products retained: 3855 of 4070 
Customers retained: 4372 of 4372 


### 1.3 Create the Revenue Attribute
Join unit price onto each transaction and compute `Revenue = Quantity * UnitPrice`.

In [8]:
transactions_clean <- transactions_clean %>%
  inner_join(products_clean %>% select(StockCode, UnitPrice), by = 'StockCode') %>%
  mutate(Revenue = Quantity * UnitPrice)

cat('Rows after price join (valid UnitPrice attached):', nrow(transactions_clean), '\n')
head(transactions_clean, 5)

Rows after price join (valid UnitPrice attached): 387877 


InvoiceNo,StockCode,CustomerID,Quantity,InvoiceDate,UnitPrice,Revenue
<chr>,<chr>,<dbl>,<dbl>,<dttm>,<dbl>,<dbl>
536365,85123A,17850,6,2010-12-01 08:26:00,2.55,15.30
536365,71053,17850,6,2010-12-01 08:26:00,3.39,20.34
536365,84406B,17850,8,2010-12-01 08:26:00,2.75,22.00
536365,84029G,17850,6,2010-12-01 08:26:00,3.39,20.34
536365,84029E,17850,6,2010-12-01 08:26:00,3.39,20.34


## Task 2: Integrate the Multiple Data Sources
Combine transactions, products, and customers into a single analysis-ready table using `dplyr` joins.

In [9]:
retail_data <- transactions_clean %>%
  left_join(products_clean %>% select(StockCode, Description), by = 'StockCode') %>%
  left_join(customers_clean, by = 'CustomerID')

dim(retail_data)

[1] 387877      9

### 2.1 Verify Dimensions and Unmatched Records
Confirm row counts after each join and check for any records that failed to match.

In [10]:
cat('transactions_clean rows:', nrow(transactions_clean), '\n')
cat('retail_data rows       :', nrow(retail_data), '\n\n')

unmatched_products  <- retail_data %>% filter(is.na(Description))
unmatched_customers <- retail_data %>% filter(is.na(Country))

cat('Transactions with no matching product description:', nrow(unmatched_products), '\n')
cat('Transactions with no matching customer/country    :', nrow(unmatched_customers), '\n')

transactions_clean rows: 387877 
retail_data rows       : 387877 

Transactions with no matching product description: 0 
Transactions with no matching customer/country    : 0 


### 2.2 Join Strategy Justification
`inner_join` was used for UnitPrice (a transaction without a valid price is unusable for revenue); `left_join` was used for Description and Country so every cleaned transaction is preserved even if enrichment data happens to be missing.

In [11]:
cat(
'- inner_join(transactions_clean, products_clean) on StockCode for UnitPrice:\n',
'  a transaction cannot generate Revenue without a valid price, so unmatched\n',
'  rows (e.g. the 215 invalid-priced stock codes) are legitimately excluded.\n\n',
'- left_join() for Description and Country:\n',
'  every cleaned+priced transaction row is KEPT even if a product description\n',
'  or customer country were missing, so no valid sale would be silently lost\n',
'  from the revenue totals. In this run all rows matched (0 unmatched), which\n',
'  confirms the CustomerID/StockCode cleaning in Task 1 already resolved\n',
'  referential consistency between the three sources.\n'
)

- inner_join(transactions_clean, products_clean) on StockCode for UnitPrice:
   a transaction cannot generate Revenue without a valid price, so unmatched
   rows (e.g. the 215 invalid-priced stock codes) are legitimately excluded.

 - left_join() for Description and Country:
   every cleaned+priced transaction row is KEPT even if a product description
   or customer country were missing, so no valid sale would be silently lost
   from the revenue totals. In this run all rows matched (0 unmatched), which
   confirms the CustomerID/StockCode cleaning in Task 1 already resolved
   referential consistency between the three sources.


## Task 3: Sales and Customer Analysis
Summarize total revenue, and rank top products, countries, and customers.

In [12]:
total_revenue <- sum(retail_data$Revenue, na.rm = TRUE)
cat('Total Sales Revenue: £', format(round(total_revenue, 2), big.mark = ','), '\n')

Total Sales Revenue: £ 10,752,840 


### 3.1 Top 5 Products by Revenue

In [13]:
top5_products <- retail_data %>%
  group_by(StockCode, Description) %>%
  summarise(TotalRevenue = sum(Revenue, na.rm = TRUE), .groups = 'drop') %>%
  arrange(desc(TotalRevenue)) %>%
  slice_head(n = 5)

top5_products

StockCode,Description,TotalRevenue
<chr>,<chr>,<dbl>
23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60
47566,PARTY BUNTING,142437.56
22423,REGENCY CAKESTAND 3 TIER,135604.80
85123A,WHITE HANGING HEART T-LIGHT HOLDER,93745.65
23166,MEDIUM CERAMIC TOP STORAGE JAR,81032.64


### 3.2 Top 5 Countries by Revenue

In [14]:
top5_countries <- retail_data %>%
  filter(!is.na(Country)) %>%
  group_by(Country) %>%
  summarise(TotalRevenue = sum(Revenue, na.rm = TRUE), .groups = 'drop') %>%
  arrange(desc(TotalRevenue)) %>%
  slice_head(n = 5)

top5_countries

Country,TotalRevenue
<chr>,<dbl>
United Kingdom,8861857.1
Netherlands,363884.5
EIRE,331660.2
Germany,263819.0
France,226975.6


### 3.3 Top 5 Customers by Purchase Value

In [15]:
top5_customers <- retail_data %>%
  group_by(CustomerID) %>%
  summarise(TotalSpend = sum(Revenue, na.rm = TRUE), .groups = 'drop') %>%
  arrange(desc(TotalSpend)) %>%
  slice_head(n = 5)

top5_customers

CustomerID,TotalSpend
<dbl>,<dbl>
18102,408760.0
14646,357531.1
17450,186038.0
14911,182690.5
16446,168472.5


### 3.4 Customer Value Segmentation
Classify every customer into Low / Medium / High / Premium value tiers using `case_when()` on total-spend quartiles.

In [16]:

customer_value <- retail_data %>%
  group_by(CustomerID) %>%
  summarise(TotalSpend = sum(Revenue, na.rm = TRUE), .groups = 'drop')

q <- quantile(customer_value$TotalSpend, probs = c(0.25, 0.5, 0.75), na.rm = TRUE)
print(q)

customer_value <- customer_value %>%
  mutate(ValueSegment = case_when(
    TotalSpend <= q[1] ~ 'Low Value',
    TotalSpend <= q[2] ~ 'Medium Value',
    TotalSpend <= q[3] ~ 'High Value',
    TRUE               ~ 'Premium'
  ))

table(customer_value$ValueSegment)

     25%      50%      75% 
 361.005  802.600 2000.640 



  High Value    Low Value Medium Value      Premium 
        1084         1085         1085         1085 

### 3.5 High-Performing vs Underperforming Market
Compare country-level revenue to identify one strong market and one weak market.

In [17]:
country_revenue <- retail_data %>%
  filter(!is.na(Country)) %>%
  group_by(Country) %>%
  summarise(TotalRevenue = sum(Revenue, na.rm = TRUE),
            NumCustomers = n_distinct(CustomerID), .groups = 'drop') %>%
  arrange(desc(TotalRevenue))

best_market  <- country_revenue %>% slice_head(n = 1)
worst_market <- country_revenue %>% slice_tail(n = 1)

cat('High-performing market:', best_market$Country,
    '- Revenue: £', format(round(best_market$TotalRevenue,2), big.mark=','),
    '- Customers:', best_market$NumCustomers, '\n')
cat('Underperforming market:', worst_market$Country,
    '- Revenue: £', round(worst_market$TotalRevenue,2),
    '- Customers:', worst_market$NumCustomers, '\n\n')
cat('Interpretation: the United Kingdom is the home market of this UK-registered\n',
    'retailer, contributing ~82% of total revenue from 3,921 active customers -\n',
    'expected concentration for a domestic online gift retailer. Saudi Arabia sits\n',
    'at the opposite extreme with a single customer and one small order, signalling\n',
    'an untapped or barely-served international market rather than a failing one.\n')

High-performing market: United Kingdom - Revenue: £ 8,861,857 - Customers: 3921 
Underperforming market: Saudi Arabia - Revenue: £ 181.44 - Customers: 1 

Interpretation: the United Kingdom is the home market of this UK-registered
 retailer, contributing ~82% of total revenue from 3,921 active customers -
 expected concentration for a domestic online gift retailer. Saudi Arabia sits
 at the opposite extreme with a single customer and one small order, signalling
 an untapped or barely-served international market rather than a failing one.


## Task 4: Store and Retrieve Data Using SQL
Persist the final integrated dataset into a SQLite database table named `retail_sales`.

In [18]:
con <- dbConnect(RSQLite::SQLite(), 'retail_analysis.db')

dbWriteTable(con, 'retail_sales', retail_data, overwrite = TRUE)

cat('Tables in database:', dbListTables(con), '\n')
cat('Rows written to retail_sales:', dbGetQuery(con, 'SELECT COUNT(*) AS n FROM retail_sales')$n, '\n')

Tables in database: retail_sales 
Rows written to retail_sales: 387877 


### 4.1 SQL Query 1 — Top 5 Customers by Revenue

In [19]:
query1 <- dbGetQuery(con, '
  SELECT CustomerID, SUM(Revenue) AS TotalRevenue
  FROM retail_sales
  GROUP BY CustomerID
  ORDER BY TotalRevenue DESC
  LIMIT 5;
')
query1

CustomerID,TotalRevenue
<dbl>,<dbl>
18102,408760.0
14646,357531.1
17450,186038.0
14911,182690.5
16446,168472.5


### 4.2 SQL Query 2 — Total Revenue by Country

In [20]:
query2 <- dbGetQuery(con, '
  SELECT Country, SUM(Revenue) AS TotalRevenue
  FROM retail_sales
  WHERE Country IS NOT NULL
  GROUP BY Country
  ORDER BY TotalRevenue DESC;
')
head(query2, 10)

dbDisconnect(con)

,Country,TotalRevenue
,<chr>,<dbl>
1,United Kingdom,8861857.13
2,Netherlands,363884.48
3,EIRE,331660.17
4,Germany,263818.97
5,France,226975.60
6,Australia,173918.61
7,Spain,67426.09
8,Switzerland,66619.97
9,Japan,48600.22


## Conclusion: Key Business Insights
Three actionable insights derived from the integrated UCI dataset and SQL analysis.

In [21]:
cat(
'1. Revenue is heavily concentrated: the UK alone contributes ~82% (£8.86M of\n',
'   £10.75M) of total revenue, and just 5 SKUs (e.g. "PAPER CRAFT, LITTLE\n',
'   BIRDIE", "PARTY BUNTING") generate over £620K combined. The business should\n',
'   prioritize inventory availability and marketing spend on these proven\n',
'   winners while treating international markets as a growth opportunity.\n\n',
'2. Customer value is highly skewed: the top 5 customers alone (e.g. CustomerID\n',
'   18102 at £408,760) contribute a disproportionate share of revenue relative\n',
'   to the ~4,372 total customers. The case_when() segmentation shows a fairly\n',
'   even split across Low/Medium/High/Premium quartiles, so a tiered loyalty or\n',
'   account-management program targeting the "Premium" segment specifically\n',
'   could meaningfully protect and grow high-value relationships.\n\n',
'3. Data quality issues were substantial in the raw feed: 135,080 transactions\n',
'   (~25%) had no CustomerID, 10,624 rows were order cancellations (negative\n',
'   quantity), and 5,429 rows were exact duplicates. Embedding this cleaning\n',
'   pipeline (Task 1) as a repeatable script before loading into the SQLite\n',
'   retail_sales table ensures future reporting stays accurate and reproducible.\n'
)

1. Revenue is heavily concentrated: the UK alone contributes ~82% (£8.86M of
    £10.75M) of total revenue, and just 5 SKUs (e.g. "PAPER CRAFT, LITTLE
    BIRDIE", "PARTY BUNTING") generate over £620K combined. The business should
    prioritize inventory availability and marketing spend on these proven
    winners while treating international markets as a growth opportunity.

 2. Customer value is highly skewed: the top 5 customers alone (e.g. CustomerID
    18102 at £408,760) contribute a disproportionate share of revenue relative
    to the ~4,372 total customers. The case_when() segmentation shows a fairly
    even split across Low/Medium/High/Premium quartiles, so a tiered loyalty or
    account-management program targeting the "Premium" segment specifically
    could meaningfully protect and grow high-value relationships.

 3. Data quality issues were substantial in the raw feed: 135,080 transactions
    (~25%) had no CustomerID, 10,624 rows were order cancellations (negative
   